# Chapter08
+ 아래 install 및 openai key입력을 진행해주세요
+ 파일이 colab 혹은 폴더 path에 넣어져 있는지 확인해주세요

In [ ]:
! pip install langchain==1.2.14 \
    langchain_openai==1.1.12 \
    langchain_community==0.4.1 \
    pymupdf \
    langchain-text-splitters \
    docling \
    langchain_chroma==1.1.0 \
    streamlit \
    rank_bm25

In [ ]:
# ! pip install langchain_openai langchain_community pymupdf langchain-text-splitters docling==2.60.0 langchain_chroma streamlit rank_bm25

In [ ]:
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


## 8.6 [프로젝트] 금융 PDF를 이용한 RAG 챗봇

In [ ]:
# 한국은행 2024년 연차보고서.pdf
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import os
import getpass

pdf_path = "한국은행 2024년 연차보고서.pdf"
converter = DocumentConverter()
result = converter.convert(source=pdf_path)
doc = result.document

[INFO] 2026-04-05 14:08:37,272 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-05 14:08:37,287 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-05 14:08:37,289 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-05 14:08:37,400 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-05 14:08:37,404 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-04-05 14:08:37,405 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-04-05 14:08:37,457 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-05 14:08:37,490 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-04-05 14:08:44,104 [RapidOCR] main.py:125: The text detection result is empty
[WARNING] 2026-04-05 14:11:20,466 [RapidOCR] main.py:125: The text detection result is empty
[WARNING] 2026-04-05 14:13:12,491 [RapidOCR] main.py:125: The text detection result is empty


In [ ]:

# 구조 기반 청킹
hc = HierarchicalChunker()
chunks = list(hc.chunk(dl_doc=doc))
print(f"청크 수: {len(chunks)}")

print(f"총 {len(chunks)}개의 청크 생성 완료.")
if chunks:
    print(chunks[0].text[:300])

# API 키 설정
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# 임베딩 모델 초기화
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

# 벡터DB 구축
vectorstore = Chroma.from_texts(
    [c.text for c in chunks],
    embedding=embedding,
    collection_name="finance_docs"
)

print(f"Chroma 컬렉션에 {vectorstore._collection.count()}개의 문서 임베딩 완료")

청크 수: 1310
총 1310개의 청크 생성 완료.
2025. 3
Chroma 컬렉션에 2620개의 문서 임베딩 완료


In [ ]:
! pip install rank_bm25

In [ ]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever

# BM25 검색기 (텍스트 유사도)
bm25_retriever = BM25Retriever.from_texts([c.text for c in chunks])
# 벡터 검색기
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
# 하이브리드 검색 결합
retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# 2. 프롬프트 정의 (최신 방식에서는 프롬프트 작성이 필수이자 핵심입니다)
system_prompt = (
    "주어진 문맥(context)을 사용하여 사용자의 질문에 답하세요. "
    "만약 답을 모른다면, 모른다고 정직하게 말하세요."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. 최신 조립형 체인 구축 (RetrievalQA 완벽 대체)
# Step A: 검색된 문서(context)를 LLM에 어떻게 밀어넣을지 결정 (stuff 방식)
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Step B: 리트리버(검색기)와 앞서 만든 체인을 결합
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# 4. 실행 및 결과 출력 (.run() 대신 .invoke() 사용)
query = "양자 간 통화 스왑 가능 국가를 전부 알려주세요"

# 최신 체인은 딕셔너리 형태로 입출력을 받습니다.
response = rag_chain.invoke({"input": query})

# 결과 출력 (정답 텍스트는 'answer' 키 안에 담겨서 옵니다)
print("🧠 답변:", response['answer'])

🧠 답변: 양자 간 통화 스왑 가능 국가는 다음과 같습니다:

1. 캐나다
2. 중국
3. 스위스
4. 일본
5. 인도네시아
6. 호주
7. UAE (아랍에미리트)
8. 말레이시아
9. 튀르키예

이 외에도 CMIM 지역 금융 협정에 포함된 ASEAN+3 13개국 및 홍콩이 있습니다.


In [ ]:
import streamlit as st

st.title("💰 금융 보고서 RAG 챗봇")
user_input = st.text_input("질문을 입력하세요:", "")

if st.button("검색"):
    if user_input:
        with st.spinner("분석 중..."):
            response = rag_chain.invoke({"input": user_input})
            answer = response["answer"]
        st.markdown(f"**답변:** {answer}")

2026-04-05 14:28:25.476 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 14:28:25.991 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-05 14:28:25.992 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 14:28:25.992 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 14:28:25.993 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 14:28:25.994 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 14:28:25.994 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 14:28:25.995 Thread 'MainThread': mi